In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv
from markitdown import MarkItDown
import glob
from pydantic import BaseModel, Field
from pathlib import Path
from litellm import completion
from tqdm import tqdm
from chromadb import PersistentClient
import numpy as np
from sklearn.manifold import TSNE
import plotly.graph_objects as go
import plotly.io as pio
from tenacity import retry, wait_exponential
import gradio as gr


In [160]:
load_dotenv(override= True)
openai = OpenAI()
api_key = os.getenv('OPENAI_API_KEY')
db_name = "vector_db"
MARKDOWN_FILES_PATH = Path("markdown_files")

In [161]:
md = MarkItDown(enable_plugins=False)
input_folder = r"database"
output_folder = "markdown_files"
os.makedirs(output_folder, exist_ok=True)

In [162]:
files = Path(input_folder).rglob("*.docx")
for file in files:
    result = md.convert(file)
    link_file = Path(file).relative_to(input_folder)
    output_path = Path(output_folder) / link_file.with_suffix(".md")
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, "w", encoding="utf-8") as f:
        f.write(result.text_content)

In [163]:
class Result(BaseModel):
    page_content: str
    metadata: dict

In [164]:
def fetch_documents():

    documents = []

    if not MARKDOWN_FILES_PATH.exists():
        raise FileNotFoundError(
            f"Folder not found: {MARKDOWN_FILES_PATH}"
        )

    for folder in MARKDOWN_FILES_PATH.iterdir():

        if not folder.is_dir():
            continue

        doc_type = folder.name

        for file in folder.rglob("*.md"):

            try:
                with open(file, "r", encoding="utf-8") as f:

                    text = f.read()

                    if text.strip():
                        documents.append({
                            "type": doc_type,
                            "source": file.as_posix(),
                            "text": text
                        })

            except Exception as e:
                print(f"Lỗi đọc file {file}: {e}")

    print(f"Loaded {len(documents)} documents")

    return documents

In [165]:
documents = fetch_documents()

Loaded 417 documents


In [166]:
print (f"đã tải {len(documents)} tài liệu")

đã tải 417 tài liệu


In [167]:
def create_chunks(documents):
    chunks = []
    CHUNK_SIZE = 1500
    OVERLAP = 200

    for document in tqdm(documents, total=len(documents)):
        text = document["text"]

        for start in range(0, len(text), CHUNK_SIZE - OVERLAP):
            chunk_text = text[start:start + CHUNK_SIZE]

            chunks.append(Result(
                page_content=chunk_text,
                metadata={
                    "source": document["source"],
                    "type": document["type"]
                }
            ))

    print(f"Created {len(chunks)} chunks")
    return chunks

In [168]:
chunks = create_chunks(documents)

100%|██████████| 417/417 [00:00<00:00, 4346.47it/s]

Created 3201 chunks


In [169]:
db_name = "vector_db"

In [170]:
def create_embeddings(chunks):
    chroma = PersistentClient(path=db_name)
    if "docs" in [c.name for c in chroma.list_collections()]:
        chroma.delete_collection("docs")

    texts = [chunk.page_content for chunk in chunks]
    
    BATCH_SIZE = 100

    all_vectors = []

    for i in range(0, len(texts), BATCH_SIZE):

        batch = texts[i:i+BATCH_SIZE]
    
        emb = openai.embeddings.create(model="text-embedding-3-large", input=batch).data
        vectors = [e.embedding for e in emb]
        all_vectors.extend(vectors)

    collection = chroma.get_or_create_collection("docs")

    ids = [str(i) for i in range(len(chunks))]
    metas = [chunk.metadata for chunk in chunks]
    
    collection.add(ids=ids, embeddings=all_vectors, documents=texts, metadatas=metas)
    print(f"Vectorstore được tạo từ {collection.count()} documents")

In [171]:
create_embeddings(chunks)

Vectorstore được tạo từ 3201 documents


In [172]:
from sklearn.preprocessing import LabelEncoder


In [173]:
chroma = PersistentClient(path=db_name)
collection = chroma.get_or_create_collection("docs")
result = collection.get(include=['embeddings', 'documents', 'metadatas'])
vectors = np.array(result['embeddings'])
documents = result['documents']
metadatas = result['metadatas']
doc_types = [metadata['type'] for metadata in metadatas]
#colors = [['blue', 'green',][['đất đai', 'công dân'].index(t)] for t in doc_types]
le = LabelEncoder()
colors = le.fit_transform(doc_types)


In [ ]:
tsne = TSNE(n_components=2, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 2D scatter plot
fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(title='hình ảnh hóa vector chroma dươi dạng 2D',
    scene=dict(xaxis_title='x',yaxis_title='y'),
    width=800,
    height=600,
    margin=dict(r=20, b=10, l=10, t=40)
)


pio.renderers.default = "browser"
fig.show()

In [ ]:
tsne = TSNE(n_components=3, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 3D scatter plot
fig = go.Figure(data=[go.Scatter3d(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    z=reduced_vectors[:, 2],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='Hình ảnh hóa vector chroma dưới dạng 3D',
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'),
    width=900,
    height=700,
    margin=dict(r=10, b=10, l=10, t=40)
)


fig.show()

In [176]:
class RankOrder(BaseModel):
    order: list[int] = Field(
        description="Danh sách xếp hạng các chunk theo độ relevance giảm dần, sử dụng chunk id."
    )

In [177]:
def rerank(question, chunks):
    system_prompt = """
Bạn là một hệ thống re-rank tài liệu.
Bạn sẽ được cung cấp: một câu hỏi và danh sách các chunk văn bản liên quan được truy vấn từ markdown_files.
Các chunk hiện đang được sắp xếp theo thứ tự truy xuất ban đầu; thứ tự này có thể đã gần đúng theo độ liên quan, nhưng bạn cần tối ưu lại.
Nhiệm vụ của bạn là:
sắp xếp lại các chunk theo mức độ liên quan tới câu hỏi,
chunk liên quan nhất đứng đầu.
Chỉ trả về danh sách các chunk id đã được sắp xếp lại, không thêm bất kỳ nội dung nào khác.
Phải bao gồm toàn bộ chunk id được cung cấp.
"""
    user_prompt = f"Người dùng đã đặt câu hỏi sau:\n\n{question}\n\nHãy sắp xếp toàn bộ các chunk văn bản theo mức độ liên quan tới câu hỏi, từ liên quan nhất đến ít liên quan nhất. IHãy sắp xếp toàn bộ các chunk văn bản theo mức độ liên quan tới câu hỏi, từ liên quan nhất đến ít liên quan nhất.\n\n"
    user_prompt += "Dưới đây là các chunk:\n\n"
    for index, chunk in enumerate(chunks):
        user_prompt += f"# CHUNK ID: {index + 1}:\n\n{chunk.page_content}\n\n"
    user_prompt += "Chỉ trả về danh sách các chunk id đã được xếp hạng, không thêm bất kỳ nội dung nào khác."
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    response = completion(model="gpt-4.1-nano", messages=messages, response_format=RankOrder)
    reply = response.choices[0].message.content
    order = RankOrder.model_validate_json(reply).order
    print(order)
    return [chunks[i - 1] for i in order]

In [178]:
RETRIEVAL_K = 10

def fetch_context_unranked(question):
    query = openai.embeddings.create(model="text-embedding-3-large", input=[question]).data[0].embedding
    results = collection.query(query_embeddings=[query], n_results=RETRIEVAL_K)
    chunks = []
    for result in zip(results["documents"][0], results["metadatas"][0]):
        chunks.append(Result(page_content=result[0], metadata=result[1]))
    return chunks

In [179]:
question = "cách làm căn cước công dân"
chunks = fetch_context_unranked(question)

In [180]:
for chunk in chunks:
    print(chunk.page_content[:50]+"...")

hị giải quyết thủ tục về căn cước (Mẫu DC02 ban hà...
). Thời hạn giải quyết: Không quá 07 ngày làm việc...
hố) trong cả nước không phụ thuộc vào nơi cư trú. ...
công dân theo các phương thức quy định tại khoản 2...
 an cấp tỉnh trong cả nước không phụ thuộc vào nơi...
èm theo Thông tư số 53/2025/TT-BCA ngày 01/7/2025 ...
ồng (Chưa quy định.) | - Thông qua Cổng dịch vụ cô...
căn cước.doc | Bản chính: 1 Bản sao: 0 |

**Bao gồ...
hính về trật tự xã hội, Bộ Công an). - Cấp lưu độn...
Mẫu DC02 ban hành kèm theo Thông tư số 17/2024/TT-...


In [181]:
reranking = rerank(question, chunks)

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]


In [182]:
for chunk in reranking:
    print(chunk.page_content[:15]+"...")

hị giải quyết t...
). Thời hạn giả...
hố) trong cả nư...
công dân theo c...
 an cấp tỉnh tr...
èm theo Thông t...
ồng (Chưa quy đ...
căn cước.doc | ...
hính về trật tự...
Mẫu DC02 ban hà...


In [183]:
question = "thông tin về làm đât đai"
RETRIEVAL_K = 10
chunks = fetch_context_unranked(question)
for index, c in enumerate(chunks):
    if "manchester" in c.page_content.lower():
        print(index)

In [184]:
reranked = rerank(question, chunks)

[2, 5, 3, 7, 1, 6, 4, 8, 9, 10]


In [185]:
def fetch_context(question):
    chunks = fetch_context_unranked(question)
    return rerank(question, chunks)

In [186]:
SYSTEM_PROMPT = """
Bạn là một chatbot hỗ trợ hành chính công, chỉ trả lời những câu hỏi liên quan đến hành chính công.
bạn đang trò chuyện với người dùng về hành chính công.
câu trả lời của bạn sẽ được đánh giá dựa trên : độ chính xác, mức độ liên quan, tính đầy đủ.
vì vậy bạn hãy đảm bảo: chỉ trả lời đúng trọng tâm câu hỏi và trả lời đầy đủ câu hỏi đó.
nếu bạn không biết câu trả lời, hãy nói rằng bạn không biết.
Để làm ngữ cảnh, dưới đây là các đoạn trích cụ thể từ Knowledge Base
có thể liên quan trực tiếp tới câu hỏi của người dùng: 
{context}
Dựa trên ngữ cảnh này, hãy trả lời câu hỏi của người dùng.
Hãy đảm bảo câu trả lời: chính xác, liên quan,đầy đủ.
"""

In [187]:
def make_rag_messages(question, history, chunks):
    context = "\n\n".join(f"Extract from {chunk.metadata['source']}:\n{chunk.page_content}" for chunk in chunks)
    system_prompt = SYSTEM_PROMPT.format(context=context)
    return [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": question}]

In [188]:
def rewrite_query(question, history=[]):
    """Hãy viết lại question của user's thành một câu hỏi cụ thể hơn, để tăng khả năng truy xuất được nội dung liên quan trong markdown_files."""
    message = f"""
Bạn đang trong một cuộc trò chuyện với user, bạn hãy trả lời những câu hỏi liên quan đến hỗ trợ hành chính công.
Bạn sắp thực hiện tra cứu thông tin trong markdown_files để trả lời câu hỏi của user.

đây là lịch sử hội thoại của user:
{history}

Và đây là câu hỏi hiện tại của user:
{question}

Chỉ trả về duy nhất một câu hỏi đã được tinh chỉnh, câu hỏi này sẽ được dùng để tìm kiếm trong markdown_files
Câu hỏi phải: rất ngắn, cụ thể, có khả năng cao nhất để truy xuất đúng nội dung liên quan, Hãy tập trung vào các chi tiết chính của câu hỏi.
QUAN TRỌNG: Chỉ trả về câu query dùng cho knowledge base, không thêm bất kỳ nội dung nào khác.
"""
    response = completion(model="gpt-4.1-nano", messages=[{"role": "system", "content": message}])
    return response.choices[0].message.content

In [189]:
rewrite_query("cách làm căn cccd kiểu gì", [])

'Cách làm căn cước công dân (CCCD) mới nhất'

In [190]:
def merge_chunks(chunks, reranked):
    merged = chunks[:]
    existing = [chunk.page_content for chunk in chunks]
    for chunk in reranked:
        if chunk.page_content not in existing:
            merged.append(chunk)
    return merged

In [191]:
def fetch_context_unranked(question):
    query = openai.embeddings.create(model="text-embedding-3-large", input=[question]).data[0].embedding
    results = collection.query(query_embeddings=[query], n_results=20)
    chunks = []
    for result in zip(results["documents"][0], results["metadatas"][0]):
        chunks.append(Result(page_content=result[0], metadata=result[1]))
    return chunks

In [192]:
def fetch_context(original_question):
    rewritten_question = rewrite_query(original_question)
    chunks1 = fetch_context_unranked(original_question)
    chunks2 = fetch_context_unranked(rewritten_question)
    chunks = merge_chunks(chunks1, chunks2)
    reranked = rerank(original_question, chunks)
    return reranked[:10]

In [ ]:
def answer_question(question: str, history: list[dict] = []) -> tuple[str, list]:
    """
    Trả lời một câu hỏi bằng RAG và trả về cả câu trả lời lẫn phần context đã được truy xuất.
    """
    chunks = fetch_context(question)
    messages = make_rag_messages(question, history, chunks)
    response = completion(model="gpt-4.1-nano", messages=messages)
    return response.choices[0].message.content, chunks